# Computational Text Analysis Part II: 
## This time it's personal

In this lecture, I'm going to introduce a few more advanced computational text analysis approaches: Topic modeling, word embeddings, and LLM-based classifications.

## Topic Modeling

The first example is topic modeling. The idea of topic modeling is to group documents (in this case, posts) which are similar to each other, and to characterize those groups somehow. In that sense, it is similar to qualitative inductive coding.

There are a number of approaches. I'm going to show you a very vanilla version of BERTopic, which is a new, fancy approach which uses a pre-trained transformer-based embedding model to understand the semantic meaning of sentences in a corpus. (These models are related to LLMs like ChatGPT, but instead of generating text, they encode text into vectors that capture meaning.)

To install it, run `conda install bertopic` in the terminal.

The BERTopic library is really great, and has [great documentation and a website here](https://maartengr.github.io/BERTopic/index.html).

**Important Note on Long Documents:** By default, BERTopic uses sentence transformers that have a 512 token limit (~300-400 words). Longer texts get truncated, meaning you only analyze the beginning. For longer documents like Reddit posts, you should either:
1. Use a longer-context embedding model (see below - recommended)
2. Intelligently truncate or chunk your documents
3. Focus only on titles or shorter sections

The first step is to load a model. (Note that `hdbscan_model = ...` line is optional, and sets some parameters which help to avoid having lots of topics. The `representation_model` is also optional, but can help to identify more representative terms for each topic. The `embedding_model` parameter lets you use models that handle longer text.)

In [ ]:
# BERTopic example
from hdbscan import HDBSCAN
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

In [ ]:
# Use a widely compatible embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

## Longer context option
# embedding_model = SentenceTransformer("BAAI/bge-m3")
# embedding_model.max_seq_length = 512

# Load a pre-trained BERTopic pipeline
hdbscan_model = HDBSCAN(min_cluster_size=25, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
representation_model = KeyBERTInspired()
topic_model = BERTopic(language="english", verbose=True, 
                       hdbscan_model=hdbscan_model, 
                       representation_model=representation_model,
                       embedding_model=embedding_model)

Let's load the subreddit data from last week. Because there are more `r/politics` posts than the other subreddits, we'll focus on those.

This next piece of code loads the data and trains the model. For speed, we'll pre-compute the embeddings separately - this allows us to reuse them if we want to adjust other parameters later. The whole process may take a few minutes to run, but BERTopic has a nice progress bar so you know that it's working.

We're also going to just look at a subset of the data for this example, to speed things up. You can always run it on the whole dataset if you have the time. 

In [ ]:
sr = pd.read_csv('https://raw.githubusercontent.com/jdfoote/Intro-to-Programming-and-Data-Science/refs/heads/master/resources/data/sr_post_data.csv')

sr = sr[sr.subreddit == 'politics']
# First we change NAs and removed/deleted to empty strings
sr.loc[(pd.isna(sr.selftext)) | (sr.selftext.isin(['[removed]', '[deleted]'])), 'selftext'] = ''
sr['all_text'] = sr.title + ' ' + sr.selftext


dataset = sr.all_text.to_list()

In [ ]:
# Pre-compute embeddings for faster processing and reusability
embeddings = embedding_model.encode(dataset, show_progress_bar=True)

In [ ]:
topics, probs = topic_model.fit_transform(dataset, embeddings=embeddings)

BERTopic decides how many topics are appropriate, and assigns each document to a topic. It also includes a "miscellaneous" topic (`Topic -1`) for documents that don't fit very well. This can include quite a few documents, as below.

In [ ]:
topic_model.get_topic_freq()

There are a number of cool visualizations and tools for understanding the topics. There are a bunch of them shown [on the BERTopic website](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html). Here are a few.

This first one visualizes the topics. We can see that they are fairly clustered.

In [ ]:
topic_model.visualize_topics()

This shows the top words and their probabilities for each of the top `n` topics

In [ ]:
topic_model.visualize_barchart(top_n_topics=15)

We can also visualize how much topics are used over time.

In [ ]:
timestamps = sr.date.to_list()
topics_over_time = topic_model.topics_over_time(dataset, timestamps, nr_bins=30, global_tuning=True)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)

### Qualitative Analyses

I think it's really vital to get back into the actual text data in order to make sure that the topics really represent what you think they do. One way to do that is to extract the documents most closely associated with each topic.

First, we get info about each document and how well it matches the topic.

In [ ]:
doc_df = topic_model.get_document_info(dataset)

# Remove the -1 topic, which is the "garbage" topic
doc_df = doc_df.loc[doc_df.Topic != -1]

Then, we sort by "Probability", which is the likelihood that the document belongs to the assigned topic, group by topic, and take the top `num_docs` documents for each topic.

I print these, but in practice what you probably want to do is save the output, so you can look at it in Excel or similar. (i.e., `top_docs.to_csv('top_documents.csv')`)

In [ ]:
num_docs = 20
top_docs = doc_df.sort_values('Probability', ascending=False).groupby('Topic').head(num_docs)
top_docs = top_docs.sort_values('Topic').loc[:, ['Topic', 'Probability', 'Document']]

In [ ]:
for topic, group in top_docs.groupby('Topic'):
    print(f"Topic {topic}")
    print(group.Document.values)
    print("\n\n")

### EXERCISE

Where topic modeling really shines is in analyzing longer texts. Identify a dataset of longer documents (e.g., news articles or paper abstracts) and apply BERTopic to it. Try out different embedding models, and see how the results change. Make sure to qualitatively analyze the topics by looking at the most representative documents for each topic.

## Word Embeddings

The next method is word embeddings. Word embeddings create a multidimensional "space" and then place words in that space based on the words that they appear near in a corpus. There are a bunch of complex versions of word embeddings, and complex uses for them. Indeed, BERTopic uses word embeddings, as do LLMs. 

The embeddings themselves can also be interesting because we can train a separate model on each community's text. Since each model learns from different language patterns, the same word may end up in a different position in the semantic space — allowing us to compare how different communities use terms differently.

One important note: unlike newer models like BERT, Word2Vec produces a single fixed vector per word regardless of the sentence it appears in. These are called *static* embeddings. The word "study" always gets the same vector, whether it appears in "study hall" or "a study found that...". This is a limitation, but it also makes the community-comparison approach work simply and interpretably.

I'm going to teach you a simple version of word embeddings called Word2Vec. In this example, we'll build the model from scratch, but another option is to use something like BERT to build on a pre-trained model.

Much of what follows is borrowed from [Laura Nelson's wonderful example](https://github.com/lknelson/DH-Institute-2017/blob/d20246758d6da88dfedbad2e75933ad4ef370930/07-Word2Vec/Word2Vec.ipynb).

We will use Laura's code as template to look at differences between some recent comments on `r/Purdue` and `r/IndianaUniversity`

In [ ]:
import numpy as np
#import pandas as pd
#from sklearn.metrics import pairwise
#from sklearn.manifold import MDS, TSNE

import gensim
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

from string import punctuation


In [ ]:

def fast_tokenize(text):
    
    # Get a list of punctuation marks
    
    lower_case = text.lower()
    
    # Iterate through text removing punctuation characters
    no_punct = "".join([char for char in lower_case if char not in punctuation])
    
    # Split text over whitespace into list of words
    tokens = no_punct.split()
    
    return tokens


In [ ]:
def tokenize_sr(df, sr_name):
    sr_data = df[df.subreddit==sr_name].body.to_list()
    sr_data = [fast_tokenize(text) for text in sr_data]
    sr_data = [text for text in sr_data if len(text) > 0]
    return sr_data

df = pd.read_csv('https://raw.githubusercontent.com/jdfoote/Intro-to-Programming-and-Data-Science/refs/heads/master/resources/data/purdue_iu_comments.csv')
purdue_data = tokenize_sr(df, 'Purdue')
iu_data = tokenize_sr(df, 'IndianaUniversity')


Word2Vec actually has two different options for algorithms. CBOW (Continuous Bag of Words) and Skip-Gram.

I won't focus on the details here. In general, CBOW is faster and does well with frequent words, while Skip-Gram can be better for rare words.

Parameters for the `gensim` `Word2Vec` function that you might want to adjust:

* vector_size: Number of dimensions for embedding model
* window: Number of context words to observe in each direction
* min_count: Words must appear this many times to be included
* max_vocab_size: Maximum number of words to consider (will remove less frequent words)
* sg (Skip-Gram): '0' indicates CBOW model; '1' indicates Skip-Gram
* alpha: Learning rate
* epochs: Number of passes (iterations) through dataset

Note: Word2Vec needs a reasonably large corpus to learn meaningful embeddings. With smaller datasets (tens of thousands of comments), you may see some noisy or surprising results. Using more `epochs` and a smaller `vector_size` can help. Also note that Word2Vec has a random component, so results will vary slightly between runs.

In [ ]:
purdue_model, iu_model = (gensim.models.Word2Vec(x, vector_size=50, window=5,
                               min_count=5, max_vocab_size=None, sg=1, alpha=0.025, epochs=20) for x in [purdue_data, iu_data])

We should now have vectors for each common word that appears in the data. Each word is represented by 50 numbers (its location in the 50-dimension meaning space)

In [ ]:
purdue_model.wv['study']

We can now do things like look at which terms are most similar to a given topic in both communities. For example, this shows the words most similar to "sports" and "studying"

In [ ]:
print(f"Purdue similar words to studying: {purdue_model.wv.most_similar('studying')}")
print(f"IU similar words to studying: {iu_model.wv.most_similar('studying')}")

In [ ]:
print(f"Purdue similar words to sports: {purdue_model.wv.most_similar('sports')}")
print(f"IU similar words to sports: {iu_model.wv.most_similar('sports')}")

### Exercise

Identify topics where you think IU and Purdue commenters might differ and figure out how to display those differences.

## Using LLMs for research

The last thing I want to show you is some example code for using LLMs (like ChatGPT or Claude) in your work.

They are incredible, flexible tools, which have a broad semantic understanding of texts, and can be used in a lot of the same ways as a trained undergraduate.

For example, let's say we wanted to identify the different hobbies that people do at each school.

In a sense, what we're doing is programming an LLM agent using natural language. So, we want to come up with a prompt. I'll show you "few-shot" prompting, which gives a few examples for the agent. This can often be helpful, especially when a task might be ambiguous. Unlike most of the programs we've written so far, you may receive different results with even small changes to a prompt. It's a stochastic process.

We'll use Purdue's RCAC (Rosen Center for Advanced Computing) to access LLMs. RCAC provides free API access to various models for Purdue students and researchers.

To get started:
1. Visit [https://www.rcac.purdue.edu/](https://www.rcac.purdue.edu/) and follow their instructions for API access
2. Get your API key from the RCAC portal
3. Save it in a file called `rcac_credentials.py` with: `api_key = "your_key_here"`

The below code uses the OpenAI-compatible API that RCAC provides.

In [ ]:
import requests
import rcac_credentials
import time
import json

# RCAC API endpoint
RCAC_API_URL = "https://genai.rcac.purdue.edu/api/chat/completions"
API_KEY = rcac_credentials.api_key

# Available models: "llama3.1:latest", "gpt-oss:120b", etc. Check RCAC docs for current options
MODEL = "gpt-oss:120b"

def get_classifications(comments, num_comments):
    """Call RCAC API to classify hobbies from comments.
    
    Args:
        comments: List of comment strings OR list of (index, comment) tuples
        num_comments: Number of comments being classified
    """
    # Handle both indexed tuples and plain strings
    if comments and isinstance(comments[0], tuple):
        # Already indexed: format with original indices
        comments_text = "\n".join([f"{idx}. {comment}" for idx, comment in comments])
    else:
        # Plain strings: format with 0-based indices
        comments_text = "\n".join([f"{i}. {comment}" for i, comment in enumerate(comments)])
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    body = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": f"""You are an AI assistant tasked with analyzing Reddit comments from Purdue University and Indiana University (IU) subreddits to identify hobbies and leisure activities.

I will provide you with {num_comments} comments. Your task is to process each one individually and return a structured JSON array.

### Input Comments (Each includes an ID and a comment):
<reddit_comments>
{comments_text}
</reddit_comments>

### Your Task:
For each comment, create a JSON object containing:
1. "id": The index number of the comment (as shown above).
2. "comment_preview": The first 5 words of the original comment (to ensure alignment).
3. "reasoning": A very brief note (5 words max) on why this is or isn't a hobby.
4. "hobby": The extracted hobby/activity, or an empty string "" if none is found.

### Extraction Rules:
- A hobby/activity is any pastime or interest (sports, clubs, gaming, etc.) other than academic studies.
- Be generous in your interpretation, but do not make assumptions (e.g., "going to class" is not a hobby; "playing intramural soccer" is).
- If multiple hobbies are mentioned, extract only the most prominent one.
- **CRITICAL**: You must return exactly {num_comments} objects in the array, in the same order as the input comments.

### Output Format:
Return ONLY a valid JSON array of objects. Do not include markdown backticks, explanations, or any text before or after the JSON.

Example:
[
  {{"id": 0, "comment_preview": "Basketball is the best sport...", "reasoning": "Mentions sports", "hobby": "watching basketball"}},
  {{"id": 1, "comment_preview": "Math 231 is super tough...", "reasoning": "Academic complaint", "hobby": ""}}
]

Begin your analysis.
"""
            }
        ],
        "stream": False
    }
    
    try:
        response = requests.post(RCAC_API_URL, headers=headers, json=body)
        if response.status_code == 200:
            result = response.json()
            classification_json = result['choices'][0]['message']['content']
            return classification_json
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
    except Exception as e:
        print(f"Error {e}: Retrying after 30 seconds")
        time.sleep(30)
        return get_classifications(comments, num_comments)


In [ ]:
get_classifications(['Test', 'I love playing basketball at Purdue', 'IU has great hiking trails'], 3)

The code below is one version of how you might do this, and is the result of running into some issues with other approaches.

I found out that if you have too many comments, then it doesn't always keep track of which is which, so I batched them into groups of 10.

Then, if it still returns the wrong number, I go one comment at a time.

Also, note that I write out comments directly to a file, and skip to where I left off. This is a good practice so you don't have to start over if there's a network error.

In [ ]:
# Get 500 comments from each subreddit
sample = df.groupby('subreddit').sample(500, random_state=2026)

# Make a list of (index, comment) tuples
comments = sample.body.to_list()
index = sample.index.to_list()
indexed_comments = list(zip(index, comments))

# Batch the comments into groups of 10
batch_size = 10
comment_batches = [indexed_comments[i:i+batch_size] for i in range(0, len(indexed_comments), batch_size)]

In [ ]:
import csv
hobbies_fn = 'hobbies.csv'
try:
    with open(hobbies_fn, 'r') as f:
        hobbies = [line.strip() for line in f]
except FileNotFoundError:
    hobbies = []
hobbies_count = len(hobbies)

batch_num = 0
with open(hobbies_fn, 'a') as f:
    out_csv = csv.writer(f)
    for batch in comment_batches:
        batch_num += 1
        batch_start = (batch_num - 1) * batch_size
        batch_end = batch_start + len(batch)
        
        # Skip batches we've already processed
        if batch_end <= hobbies_count:
            continue
            
        # If we're partway through a batch, only process remaining comments
        if batch_start < hobbies_count:
            skip = hobbies_count - batch_start
            batch = batch[skip:]
            print(f"Resuming batch {batch_num} at comment {skip + 1}")

        print(f"Processing batch {batch_num} of {len(comment_batches)} ({len(batch)} comments)")
        
        # Pass batch with original indices to LLM
        response_text = get_classifications(batch, len(batch))
        print(response_text)
        print(batch)
        
        try:
            curr_hobbies = json.loads(response_text)
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            print(f"Response was: {response_text}")
            curr_hobbies = [{"id": original_idx, "hobby": ""} for original_idx, _ in batch]
        
        # Check if we got the right number of items and all are dicts
        if len(curr_hobbies) != len(batch) or not all(isinstance(item, dict) for item in curr_hobbies):
            print(f"Warning: Expected {len(batch)} dict objects but got {len(curr_hobbies)} items or wrong types. Processing individually...")
            curr_hobbies = []
            for original_idx, comment in batch:
                response_text = get_classifications([comment], 1)
                try:
                    hobby_list = json.loads(response_text)
                    # Extract hobby from the response (handle both dict and string responses)
                    if hobby_list and isinstance(hobby_list[0], dict):
                        hobby = hobby_list[0].get("hobby", "")
                    else:
                        hobby = ""
                except (json.JSONDecodeError, IndexError, TypeError):
                    print(f"Failed to parse response for comment {original_idx}: {response_text}")
                    hobby = ""
                curr_hobbies.append({"id": original_idx, "hobby": hobby})
        
        # Write results to CSV (sorted by id to maintain order)
        for hobby_obj in curr_hobbies:
            if isinstance(hobby_obj, dict):
                id = hobby_obj.get("id", "")
                comment_preview = hobby_obj.get("comment_preview", "")
                hobby = hobby_obj.get("hobby", "")
            else:
                hobby = ""
            out_csv.writerow([id, comment_preview, hobby])

We can then put the hobbies back into the original dataframe, and do things like filter by them, compare them across campuses, etc.

In [ ]:
sample

In [ ]:
hobbies_df = pd.read_csv(hobbies_fn, names=["id", "comment_preview", "hobby"])
sample_with_hobbies = sample.merge(hobbies_df, left_index=True, right_on="id")

In [ ]:
sample_with_hobbies.to_csv('purdue_iu_comments_hobbies.csv', index=False)

In [ ]:
sample_with_hobbies[sample_with_hobbies.hobby != ''].groupby('subreddit').hobby.count()